In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 31.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9


In [1]:
!pip -q install datasets pandas tqdm transformers accelerate bitsandbytes sentencepiece huggingface_hub

# 1. Charger le modèle

### 1.1 Avec huggingface (local & sans crédits)

In [2]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig,
    GenerationConfig,
)


MODEL_ID =  "Qwen/Qwen2.5-3B-Instruct"  # alternative à GPT4o

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # important sur colab lorsqu'on fait beaucoup de génération

# quantization 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# Evite les warnings pad/eos
tok.pad_token = tok.eos_token
model.config.pad_token_id = tok.eos_token_id

gen = pipeline("text-generation", model=model, tokenizer=tok)

gen_config = GenerationConfig(
    do_sample=True,
    temperature=0.7,
    max_new_tokens=64,
    pad_token_id=tok.eos_token_id,
)



Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:

prompt_template = lambda job_description: f"""Read the following job description and create a concise job search query with at most 3 specialized
skills or areas of expertise that are distinct to the role. Exclude generic data science or software engineering skills like AI,
machine learning, and coding languages unless they are explicitly highlighted as unique or advanced.
Keep the query short and human-like, suitable for typing into a search engine.

Here's the job description: {job_description}"""

def clean_query(s: str) -> str:
    s = (s or "").strip().replace('"', "")
    # garde la première ligne si le modèle “bavarde”
    s = s.split("\n")[0].strip()
    # retire puces éventuelles
    s = s.lstrip("-•").strip()
    return s


def generate_queries(job_descriptions, batch_size=8, max_chars=4000):
    prompts = [
        f"<|user|>\n{prompt_template(...)}\n<|assistant|>\n"
        for jd in job_descriptions
    ]

    results = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i : i + batch_size]
        outs = gen(
            batch,
            generation_config=gen_config,
            return_full_text=False,
        )
        # outs = list; chaque élément est une liste de dicts (top-k sequences)
        for o in outs:
            results.append(clean_query(o[0]["generated_text"]))
    return results



### 1.2 Avec huggingface inférence client (local & avec crédits)

In [1]:
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from google.colab import userdata

from huggingface_hub import InferenceClient

hf_token = userdata.get("HF_TOKEN")
client = InferenceClient(token=hf_token)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

prompt_template = lambda job_description : f"""Read the following job description and create a concise job search query with at most 3 specialized skills or \
areas of expertise that are distinct to the role. Exclude generic data science or software engineering skills like AI, machine \
learning, and coding languages unless they are explicitly highlighted as unique or advanced. Keep the query short and human-like, \
suitable for typing into a search engine.

Here's the job description: {job_description}"""

def generate_query(job_description: str) -> str:
    prompt = prompt_template(job_description)

    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=80,
    )
    return resp.choices[0].message.content.strip().replace('"', "")



### 1.3 Avec Azure openAI

In [ ]:
import pandas as pd
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
import json
from datasets import load_dataset
from google.colab import userdata
from openai import AzureOpenAI


client = AzureOpenAI(
    azure_endpoint=userdata.get('AZURE_ENDPOINT'),
    api_key=userdata.get('AZURE_OPENAI_KEY'),
    api_version="2023-12-01-preview",
)

prompt_template = lambda job_description : f"""Read the following job description and create a concise job search query with at most 3 specialized skills or \
areas of expertise that are distinct to the role. Exclude generic data science or software engineering skills like AI, machine \
learning, and coding languages unless they are explicitly highlighted as unique or advanced. Keep the query short and human-like, \
suitable for typing into a search engine.

Here's the job description: {job_description}"""


def generate_query(job_description):
    """
        Fonction permettant de générer une requête synthétique pour saisir la description du poste.
    """

    # gérérer le prompt
    prompt = prompt_template(job_description)

    # faire le call api
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature = 0.7
    )

    # retourner la réponse
    return response.choices[0].message.content

# 2. Charger les données

In [ ]:
# Charger les données de HF
ds = load_dataset("datastax/linkedin_job_listings")

# convertir dans un dataframe pandas
df = ds['train'].to_pandas()

# garder uniquement les titres et descriptions
df = df[['title', 'description']]
df.shape  # dataset volumineux

postings.csv:   0%|          | 0.00/517M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/123849 [00:00<?, ? examples/s]

(123849, 2)

In [ ]:

# Liste des éléments textuels recherchés
search_terms = [
    "Data Scientist",
    "Data Analyst",
    "Machine Learning Engineer",
    "Data Engineer",
    "AI Engineer",
    "Deep Learning",
]
# Créer un modèle d'expression régulière pour correspondre à n'importe laquelle des chaînes
pattern = "|".join(search_terms)

# Filtrer les lignes qui contiennent l'un des termes de recherche
df = df[df["title"].str.contains(pattern, case=False, na=False)].copy()

# réduire le dataset
df = df.iloc[:1200].copy()
df.shape

(1179, 2)

In [ ]:
# sauvegarder le fichier
df.to_csv('job_data.csv')

# 3. Générer des requêtes synthétiques

In [ ]:
job_description_list = df['description'].fillna("").to_list()

### 3.1 Pour huggingface avec modèle local & sans crédits

In [ ]:
synthetic_query_list = generate_queries(
    job_description_list,
    batch_size=6,   # petit pour ne pas faire sauter le notebook colab
    max_chars=3000,
)


df["query"] = synthetic_query_list
df.to_csv("job_data_w_query.csv", index=False)

df.head(5)

### 3.2 Pour AzureOpenAI & huggingface avec crédits

Approche sans patch (huggingface & AzureOpenAI) :

In [ ]:
# Approche sans patch :
from tqdm import tqdm

synthetic_query_list = []
for job_description in tqdm(job_description_list):
    synthetic_query_list.append(generate_query(job_description).replace('"',''))

# Rajouter les requêtes à df
df['query'] = synthetic_query_list

df.to_csv('job_data_w_query.csv')

approche par batch (Uniquement AzureOpenAI):

In [ ]:
job_description_list = df['description'].to_list()


# Créer des requêtes par batch
batch_requests = [
    {
        "custom_id": f"request-{i+1}",  # Identifiant personnalisé pour le suivi
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o",
            "messages": [
                {"role": "user", "content": prompt_template(job_description)}
            ],
            "temperature": 0.7
        }
    }
    for i, job_description in enumerate(job_description_list)
]


# Convertir au format JSONL (JSON délimité par des sauts de ligne)
batch_jsonl = "\n".join(json.dumps(request) for request in batch_requests)

In [ ]:
# Enregistrer dans un fichier .jsonl
with open("batch_requests.jsonl", "w") as file:
    file.write(batch_jsonl)

In [ ]:


# upload file using the file_client
batch_input_file = client.files.create(
    file=open("batch_requests.jsonl", "rb"),
    purpose="fine-tune"
)

print(batch_input_file)

FileObject(id='file-a7a681ea09e34dafaabb15b637dffd73', bytes=5041930, created_at=1753879178, filename='batch_requests.jsonl', object='file', purpose='fine-tune', status='pending', expires_at=None, status_details=None, updated_at=1753879178)
